# Studio Framework Healthchecks

Purpose: verify the local CCoP 2.0 evaluation stack is healthy before running experiments.

**What this notebook does**
1. Load configuration from `src/config/.env.local`
2. Verify Apple Silicon MPS device availability (set as default for inference)
3. Print bucketed configuration
4. Run `start_local.sh --status` — if degraded, run `start_local.sh` to bring the stack up
5. Run end-to-end smoke test (`--status --deep`) — exercises hybrid retrieval + Ollama generation

**Kernel:** `studio-ssdlc (poetry)` — exposes the project venv (torch, dotenv, project modules).


In [1]:
import os
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path(subprocess.check_output(
    ["git", "rev-parse", "--show-toplevel"], text=True
).strip())
SRC_DIR = REPO_ROOT / "src"
ENV_FILE = SRC_DIR / "config" / ".env.local"
START_SCRIPT = SRC_DIR / "scripts" / "start_local.sh"

print(f"Repo root:    {REPO_ROOT}")
print(f"Src dir:      {SRC_DIR}")
print(f"Env file:     {ENV_FILE}  (exists={ENV_FILE.exists()})")
print(f"Startup:      {START_SCRIPT}  (exists={START_SCRIPT.exists()})")
print(f"Python:       {sys.executable}")


Repo root:    /Users/sagarpratapsingh/dev/sagerstack/studio-ssdlc
Src dir:      /Users/sagarpratapsingh/dev/sagerstack/studio-ssdlc/src
Env file:     /Users/sagarpratapsingh/dev/sagerstack/studio-ssdlc/src/config/.env.local  (exists=True)
Startup:      /Users/sagarpratapsingh/dev/sagerstack/studio-ssdlc/src/scripts/start_local.sh  (exists=True)
Python:       /Users/sagarpratapsingh/dev/sagerstack/studio-ssdlc/src/.venv/bin/python


## 1. Load `.env.local` and print bucketed configuration


In [2]:
from dotenv import load_dotenv

loaded = load_dotenv(ENV_FILE, override=True)
print(f"Loaded {ENV_FILE.name}: {loaded}")
print()

BUCKETS = {
    "Ollama / Model": [
        "CCOP_OLLAMA_HOST", "CCOP_OLLAMA_TIMEOUT", "CCOP_MODEL_NAME",
        "CCOP_MODEL_HF_REPO", "CCOP_MODEL_QUANTIZATION", "CCOP_MODEL_CACHE_DIR",
        "CCOP_CONTEXT_LENGTH", "CCOP_DEFAULT_TEMPERATURE",
        "CCOP_DEFAULT_TOP_P", "CCOP_DEFAULT_TOP_K", "CCOP_DEFAULT_MAX_TOKENS",
    ],
    "Qdrant (Vector Store)": [
        "CCOP_QDRANT_URL", "CCOP_QDRANT_COLLECTION_NAME",
        "CCOP_QDRANT_EMBEDDING_MODEL", "CCOP_QDRANT_SPARSE_MODEL",
    ],
    "RAG Pipeline": [
        "CCOP_RAG_GRADING_ENABLED", "CCOP_RAG_SIMILARITY_THRESHOLD",
        "CCOP_RAG_RETRIEVAL_TOP_K",
    ],
    "Paths": [
        "CCOP_TEST_CASES_DIR", "CCOP_RESULTS_DIR", "CCOP_LOG_FILE",
    ],
    "LLM Judge": [
        "CCOP_LLM_JUDGE_MODEL", "CCOP_CLAUDE_CLI_TIMEOUT",
    ],
    "Databricks (legacy)": [
        "CCOP_DATABRICKS_HOST", "CCOP_DATABRICKS_CATALOG", "CCOP_DATABRICKS_SCHEMA",
        "CCOP_DATABRICKS_VECTOR_SEARCH_ENDPOINT", "CCOP_DATABRICKS_EMBEDDING_ENDPOINT",
    ],
    "Evaluation": [
        "CCOP_MAX_CONCURRENT_EVALUATIONS", "CCOP_MOCK_MODE",
    ],
    "Logging": [
        "CCOP_LOG_LEVEL", "CCOP_LOG_FORMAT", "CCOP_DEBUG",
    ],
}

SENSITIVE = {"CCOP_DATABRICKS_TOKEN", "CCOP_ANTHROPIC_API_KEY"}

def _mask(key: str, val: str) -> str:
    if key in SENSITIVE and val and val != "<unset>":
        return val[:4] + "..." + val[-4:] if len(val) > 8 else "***"
    return val

for bucket, keys in BUCKETS.items():
    print(f"-- {bucket} --")
    for k in keys:
        v = os.environ.get(k, "<unset>")
        print(f"  {k:<42} = {_mask(k, v)}")
    print()


Loaded .env.local: True

-- Ollama / Model --
  CCOP_OLLAMA_HOST                           = http://localhost:11434
  CCOP_OLLAMA_TIMEOUT                        = 300
  CCOP_MODEL_NAME                            = primus-reasoning
  CCOP_MODEL_HF_REPO                         = trendmicro-ailab/Llama-Primus-Reasoning
  CCOP_MODEL_QUANTIZATION                    = Q5_K_M
  CCOP_MODEL_CACHE_DIR                       = ~/.cache/ccop-models
  CCOP_CONTEXT_LENGTH                        = 4096
  CCOP_DEFAULT_TEMPERATURE                   = 0.7
  CCOP_DEFAULT_TOP_P                         = 0.9
  CCOP_DEFAULT_TOP_K                         = 40
  CCOP_DEFAULT_MAX_TOKENS                    = 1024

-- Qdrant (Vector Store) --
  CCOP_QDRANT_URL                            = http://localhost:6333
  CCOP_QDRANT_COLLECTION_NAME                = ccop_clauses_hybrid
  CCOP_QDRANT_EMBEDDING_MODEL                = BAAI/bge-large-en-v1.5
  CCOP_QDRANT_SPARSE_MODEL                   = Qdrant/bm25

-- RAG Pi

## 2. Torch / MPS device setup

Set `mps` as the default device for inference. Also export `PYTORCH_ENABLE_MPS_FALLBACK=1` so ops unsupported on MPS fall back to CPU rather than erroring.


In [3]:
import torch

print("-- Torch / Device --")
print(f"  torch version:      {torch.__version__}")
print(f"  mps available:      {torch.backends.mps.is_available()}")
print(f"  mps built:          {torch.backends.mps.is_built()}")
print(f"  cuda available:     {torch.cuda.is_available()}")

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    torch.set_default_device("mps")
    os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
    print(f"  -> default device:  mps  (PYTORCH_ENABLE_MPS_FALLBACK={os.environ['PYTORCH_ENABLE_MPS_FALLBACK']})")
else:
    DEVICE = torch.device("cpu")
    print("  -> default device:  cpu  (MPS unavailable)")

# Sanity: tiny matmul on the selected device
x = torch.randn(4, 4, device=DEVICE)
y = (x @ x.T).sum().item()
print(f"  device sanity matmul sum: {y:.4f}")


-- Torch / Device --
  torch version:      2.7.1
  mps available:      True
  mps built:          True
  cuda available:     False
  -> default device:  mps  (PYTORCH_ENABLE_MPS_FALLBACK=1)


  device sanity matmul sum: 17.2660


## 3. Stack status

Call `start_local.sh --status`. Exit code:
- **0** = all checks green, stack ready
- **1** = at least one degraded component (ingestion or service down)


In [ ]:
# Streaming runner: streams stdout (stderr merged) in real time, returns buffered result.
from collections import namedtuple as _namedtuple

RunResult = _namedtuple("RunResult", ["stdout", "returncode"])

def run_script(args, timeout=None):
    proc = subprocess.Popen(
        [str(START_SCRIPT), *args],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    buf = []
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
            buf.append(line)
        proc.wait(timeout=timeout)
    except KeyboardInterrupt:
        proc.terminate()
        proc.wait(timeout=10)
        raise
    return RunResult(stdout="".join(buf), returncode=proc.returncode)


result = run_script(["--status"])
print(f"\nexit_code = {result.returncode}")
status_ok = result.returncode == 0


## 4. Bring up the stack (if degraded)

Idempotent. If a Qdrant collection is missing or empty, this runs the full ingestion (several minutes).


In [ ]:
if status_ok:
    print("Stack already healthy - skipping startup.")
else:
    print("Stack degraded - running startup (first-run ingestion can take several minutes)...\n")
    result = run_script([], timeout=1800)
    print(f"\nstartup exit_code = {result.returncode}")


## 5. End-to-end smoke test (`--deep`)

Exercises the full stack:
- **Retrieval smoke** — `ccop-eval query ask ... --mode rag-only` → hits Qdrant + embeddings + RAG graph
- **Generation smoke** — `curl $OLLAMA_HOST/api/generate` with `num_predict=4`, 30s timeout


In [ ]:
result = run_script(["--status", "--deep"], timeout=300)
print(f"\nsmoke exit_code = {result.returncode}")
smoke_ok = result.returncode == 0


## 6. Summary


In [7]:
print("-- Studio Framework Healthcheck Summary --")
print(f"  Config loaded from: {ENV_FILE}")
print(f"  Torch device:       {DEVICE}")
print(f"  Stack status check: {'PASS' if status_ok else 'ran startup'}")
print(f"  Deep smoke test:    {'PASS' if smoke_ok else 'FAIL'}")
print()
if smoke_ok:
    print("Stack is healthy. Ready for experiments.")
else:
    print("Stack is NOT healthy. See output above for failing checks.")


-- Studio Framework Healthcheck Summary --
  Config loaded from: /Users/sagarpratapsingh/dev/sagerstack/studio-ssdlc/src/config/.env.local
  Torch device:       mps
  Stack status check: PASS
  Deep smoke test:    PASS

Stack is healthy. Ready for experiments.
